# Data pipeline quickstart

Runs the `multimodal_ad.data` pipeline end to end on a small, **deterministic synthetic dataset**: no OASIS-3 access needed, no manual paths to edit. Generates fake NIfTI volumes, builds and validates a typed manifest, does a subject-wise train/test split, processes one scan, and visualizes its central slice.

This is a thin wrapper: all logic lives in `multimodal_ad.data` (see [`src/multimodal_ad/data/__init__.py`](../src/multimodal_ad/data/__init__.py)); this notebook only calls it and displays results. Equivalent CLI:
`uv run python -m multimodal_ad.cli`.

In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt

from multimodal_ad.data.manifest import ScanManifest
from multimodal_ad.data.splits import subject_train_test_split
from multimodal_ad.data.synthetic import SyntheticDatasetConfig, generate_synthetic_dataset
from multimodal_ad.data.volumes import ProcessingConfig, process_scan

output_dir = Path(tempfile.mkdtemp(prefix="multimodal-ad-quickstart-"))
output_dir

## Generate a deterministic synthetic dataset

Same seed, same subject count -> byte-identical output every run.

In [ ]:
config = SyntheticDatasetConfig(n_subjects=6, seed=1234)
manifest = generate_synthetic_dataset(output_dir, config)
len(manifest)

## Validate the manifest

`ScanManifest.from_csv` re-validates the data contract (columns, modality, labels, file existence) on read.

In [ ]:
reloaded = ScanManifest.from_csv(output_dir / "manifest.csv")
len(reloaded), reloaded.records[0]

## Subject-wise train/test split

`subject_train_test_split` keeps every session for a given subject on one side, so no subject leaks across the split.

In [ ]:
df = reloaded.to_dataframe()
train_df, test_df = subject_train_test_split(df, seed=1234)
len(train_df), len(test_df)

## Process one scan and look at its central slice

`process_scan` normalizes, crops to the detected brain bounding box, and resizes to a common frame size.

In [ ]:
processing_config = ProcessingConfig(n_frames=16, image_size=64)
record = ScanManifest.from_dataframe(train_df).records[0]
volume = process_scan(record.file_path, record.modality, processing_config)
volume.shape, float(volume.min()), float(volume.max())

In [ ]:
central_slice = volume[:, :, volume.shape[-1] // 2]
plt.imshow(central_slice, cmap="gray")
plt.title(f"{record.session_id} (central slice)")
plt.axis("off")
plt.show()

## Next steps

- This dataset is synthetic and has no clinical meaning; it only exercises the same code paths real OASIS-3 data would go through.
- For the model pipeline on similarly tiny synthetic data, see `02-tiny-model-workflow.ipynb`.
- Real data access, contracts, and the OASIS-3 adapter are documented in `docs/data-access.md`.